In [ ]:
from google.colab import files
uploaded = files.upload()

Saving data.csv to data.csv


In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter

In [ ]:
df = pd.read_csv("/content/data.csv")

df.head(10)

,timestamp,weight,weight_status,weight_status_score,is_stable,wagon_event,wagon_event_conf,wagon_event_secondary,wagon_event_conf_secondary,ocr_detection_side1,ocr_detection_side2,rake_direction
0,2026-03-14 15:30:35,0.0,weighing_in,0.929,False,single_wagon,0.965870,single_wagon,0.998839,NaN,NaN,inwards
1,2026-03-14 15:30:35,0.0,weighing_in,0.929,False,single_wagon,0.973923,single_wagon,0.991813,NaN,NaN,inwards
2,2026-03-14 15:30:35,0.0,weighing_in,0.928,False,single_wagon,0.973769,single_wagon,0.991813,NaN,NaN,inwards
3,2026-03-14 15:30:35,0.0,weighing---,0.883,False,single_wagon,0.973769,single_wagon,0.999154,NaN,NaN,inwards
4,2026-03-14 15:30:35,0.0,weighing---,0.883,False,single_wagon,0.980808,single_wagon,0.999770,NaN,NaN,inwards
5,2026-03-14 15:30:35,0.0,weighing---,0.882,False,single_wagon,0.980808,single_wagon,0.999770,NaN,NaN,inwards
6,2026-03-14 15:30:35,0.0,weighing---,0.882,False,single_wagon,0.971536,single_wagon,0.999819,NaN,NaN,inwards
7,2026-03-14 15:30:35,0.0,weighing---,0.891,False,single_wagon,0.971536,single_wagon,0.999819,NaN,NaN,inwards
8,2026-03-14 15:30:35,0.0,weighing---,0.891,False,single_wagon,0.858810,single_wagon,0.999997,NaN,NaN,inwards
9,2026-03-14 15:30:35,0.0,weighing_over,0.932,False,single_wagon,0.858810,single_wagon,0.999998,NaN,NaN,inwards


**Cleaning The  Data**

In [ ]:
df = df.drop_duplicates()
df = df.reset_index(drop=True)

**Createing the  OCR Cleaning Function**

In [ ]:
def clean_ocr(text):
    if pd.isna(text):
        return None

    # keep only digits
    text = re.sub(r'\D', '', str(text))

    # ignore very short numbers
    if len(text) < 6:
        return None

    return text

**Get The Best Wagon Number**

In [ ]:
def get_best_ocr(ocr_list):
    if len(ocr_list) == 0:
        return "UNKNOWN"

    counter = Counter(ocr_list)
    return counter.most_common(1)[0][0]

**Initializeing the Variables**

In [ ]:
ocr_buffer = []        # stores OCR readings
results = []           # final output

last_status = None     # previous weight status
transition_count = 0   # count transitions

wagon_present = False  # is wagon visible?

**The Main Logic Of All Working**

In [ ]:
for i, row in df.iterrows():

    # -------------------
    # 1. READ OCR
    # -------------------
    ocr1 = clean_ocr(row['ocr_detection_side1'])
    ocr2 = clean_ocr(row['ocr_detection_side2'])

    if ocr1:
        ocr_buffer.append(ocr1)
    if ocr2:
        ocr_buffer.append(ocr2)


    # -------------------
    # 2. DETECT WAGON
    # -------------------
    if (row['wagon_event'] == "single_wagon" and row['wagon_event_conf'] > 0.8) or \
       (row['wagon_event_secondary'] == "single_wagon" and row['wagon_event_conf_secondary'] > 0.8):

        wagon_present = True


    # -------------------
    # 3. DETECT WEIGHT
    # -------------------
    status = row['weight_status']

    # detect transition
    if last_status == "weighing---" and status == "weighing_over":
        transition_count += 1

        # every 4 transitions = real weight
        if transition_count == 4:
            transition_count = 0

            if row['is_stable'] == True and row['weight'] > 0:

                wagon_number = get_best_ocr(ocr_buffer)

                results.append({
                    "wagon_number": wagon_number,
                    "weight": row['weight']
                })

                # reset for next wagon
                ocr_buffer = []
                wagon_present = False

    last_status = status

In [ ]:
output = pd.DataFrame(results)

output.head(11)

,wagon_number,weight
0,224422101820,87.4
1,224422103350,85.8
2,222622105640,82.2
3,224422105570,82.8
4,224422105570,79.2
5,224422103420,82.6
6,221622104960,82.4
7,224422101060,73.2
8,222622101060,80.8
9,224422100760,83.6


In [ ]:
output.to_csv("output.csv", index=False)